# ERRANT で英語の訂正を誤りタイプ付きの編集に変換する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/18_gec_education/errant.ipynb)

NLP クックブック 18-03 ／ 文法誤り訂正と言語教育

英語の誤り文と訂正文から編集を抽出し、「動詞の時制」「冠詞」「綴り」などの誤りタイプを自動で付けたい。

In [1]:
!pip install -q errant
!python -m spacy download en_core_web_sm -q

In [2]:
import errant

annotator = errant.load("en")
orig = annotator.parse("I has went to school yesterday and buyed a apple , but it were not delicious .")
cor = annotator.parse("I went to school yesterday and bought an apple , but it was not delicious .")
edits = annotator.annotate(orig, cor)
for e in edits:
    print(f"{e.o_start:2}-{e.o_end:2}  {e.o_str or '∅':8} → {e.c_str or '∅':8} {e.type}")

 1- 2  has      → ∅        U:VERB:TENSE
 7- 8  buyed    → bought   R:VERB:INFL
 8- 9  a        → an       R:DET
13-14  were     → was      R:VERB:SVA


In [3]:
# M2 形式 (GEC の標準的なアノテーション形式) で出力
print("S", orig.text)
for e in edits:
    print(e.to_m2())

S I has went to school yesterday and buyed a apple , but it were not delicious . 
A 1 2|||U:VERB:TENSE||||||REQUIRED|||-NONE-|||0
A 7 8|||R:VERB:INFL|||bought|||REQUIRED|||-NONE-|||0
A 8 9|||R:DET|||an|||REQUIRED|||-NONE-|||0
A 13 14|||R:VERB:SVA|||was|||REQUIRED|||-NONE-|||0


- タイプは「操作:品詞:詳細」。操作は M (Missing、脱落)、U (Unnecessary、余剰)、R (Replacement、置換)。
- 入力は空白でトークン化済みの文を想定している。句読点の前に空白を入れる。
- ERRANT は BEA-2019 Shared Task の公式評価ツールでもある。コマンドラインでは `errant_parallel` で M2 を作り、`errant_compare` でスコアを計算する。